In [1]:
import pandas as pd
import numpy as np
import pyfixest as pf

panel = pd.read_csv("../data/processed/panel_gdp.csv")
expo = pd.read_csv("../data/processed/exposure_isco2.csv")

d = panel[panel["main_sample"]].merge(expo, on="isco2", how="left")
d = d.dropna(subset=["ln_s_may", "ln_e_2025"]).copy()
d["country_id"] = d["geo"].astype("category").cat.codes
print(len(d), d["X_human"].isna().sum())

482 0


In [2]:
d["lnE_x_X"] = d["ln_e_2025"] * d["X_human"]
mA = pf.feols("ln_s_may ~ ln_e_2025 + lnE_x_X | geo + isco2", data=d, vcov={"CRV1": "country_id"})
mA.summary()
print(mA.wildboottest(param="lnE_x_X", reps=9999, seed=42))

d["lnE_x_Xg"] = d["ln_e_2025"] * d["X_gpt"]
mA_gpt = pf.feols("ln_s_may ~ ln_e_2025 + lnE_x_Xg | geo + isco2", data=d, vcov={"CRV1": "country_id"})
mA_gpt.summary()

###

Estimation:  OLS
Dep. var.: ln_s_may, Fixed effects: geo + isco2
sample: None = all
Inference:  CRV1
Observations:  482

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| ln_e_2025     |      0.033 |        0.067 |     0.491 |      0.630 | -0.109 |   0.175 |
| lnE_x_X       |      0.009 |        0.170 |     0.056 |      0.956 | -0.353 |   0.372 |
---
RMSE: 0.221 R2: 0.982 R2 Within: 0.005 
param                         lnE_x_X
t value           0.05483718531915055
Pr(>|t|)           0.9582958295829583
bootstrap_type                     11
inference             CRV(country_id)
impose_null                      True
ssc                           1.19596
dtype: object
###

Estimation:  OLS
Dep. var.: ln_s_may, Fixed effects: geo + isco2
sample: None = all
Inference:  CRV1
Observations:  482

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>

In [3]:
raw = pd.read_csv("../data/raw/anthropic/aei_claude_ai_2026-06-26.csv",
                  usecols=["geo_id", "geo_level", "category_name", "metric_id", "date_start", "value"])
w = raw[(raw["geo_level"] == "country") & (raw["category_name"] == "overall") &
        (raw["metric_id"] == "use_case_work_pct") & (raw["date_start"] == "2026-05-01")].copy()

codes = {"AUT": "AT", "BEL": "BE", "CZE": "CZ", "DNK": "DK", "FIN": "FI", "FRA": "FR",
         "DEU": "DE", "GRC": "EL", "IRL": "IE", "ITA": "IT", "NLD": "NL", "POL": "PL",
         "PRT": "PT", "ROU": "RO", "ESP": "ES", "SWE": "SE",
         "BGR": "BG", "HUN": "HU", "LTU": "LT", "SVK": "SK"}
w = w[w["geo_id"].isin(codes)].copy()
w["geo"] = w["geo_id"].map(codes)
w = w[["geo", "value"]].rename(columns={"value": "work_pct"})

d = d.merge(w, on="geo", how="left")
print(d["work_pct"].isna().sum())
d.groupby("geo")["work_pct"].first().sort_values()

0


geo
BE    32.84
FR    34.23
DK    35.36
IE    36.26
AT    37.20
DE    37.62
SE    38.11
FI    38.49
RO    38.96
CZ    39.61
NL    40.82
PL    41.77
ES    42.28
IT    42.89
PT    43.80
EL    47.13
Name: work_pct, dtype: float64

In [4]:
d["W10"] = (d["work_pct"] - d["work_pct"].mean()) / 10
d["lnE_x_W"] = d["ln_e_2025"] * d["W10"]
mB = pf.feols("ln_s_may ~ ln_e_2025 + lnE_x_W | geo + isco2", data=d, vcov={"CRV1": "country_id"})
mB.summary()
print(mB.wildboottest(param="lnE_x_W", reps=9999, seed=42))

###

Estimation:  OLS
Dep. var.: ln_s_may, Fixed effects: geo + isco2
sample: None = all
Inference:  CRV1
Observations:  482

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| ln_e_2025     |      0.038 |        0.035 |     1.081 |      0.297 | -0.037 |   0.112 |
| lnE_x_W       |     -0.027 |        0.017 |    -1.558 |      0.140 | -0.064 |   0.010 |
---
RMSE: 0.221 R2: 0.982 R2 Within: 0.006 
param                         lnE_x_W
t value           -1.5319205745246316
Pr(>|t|)           0.0855085508550855
bootstrap_type                     11
inference             CRV(country_id)
impose_null                      True
ssc                           1.19596
dtype: object


In [5]:
regulated = ["OC22", "OC32", "OC24", "OC33", "OC26"]
d["R"] = d["isco2"].isin(regulated).astype(int)
d["lnE_x_R"] = d["ln_e_2025"] * d["R"]
print(sorted(d.loc[d["R"] == 1, "isco2"].unique()))

mC = pf.feols("ln_s_may ~ ln_e_2025 + lnE_x_X + lnE_x_R | geo + isco2", data=d, vcov={"CRV1": "country_id"})
mC.summary()
print(mC.wildboottest(param="lnE_x_R", reps=9999, seed=42))

['OC22', 'OC24', 'OC26', 'OC32', 'OC33']
###

Estimation:  OLS
Dep. var.: ln_s_may, Fixed effects: geo + isco2
sample: None = all
Inference:  CRV1
Observations:  482

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| ln_e_2025     |      0.041 |        0.070 |     0.583 |      0.569 | -0.108 |   0.190 |
| lnE_x_X       |      0.010 |        0.172 |     0.058 |      0.954 | -0.356 |   0.376 |
| lnE_x_R       |     -0.073 |        0.050 |    -1.459 |      0.165 | -0.181 |   0.034 |
---
RMSE: 0.221 R2: 0.982 R2 Within: 0.007 
param                         lnE_x_R
t value           -1.4340425247978925
Pr(>|t|)           0.1272127212721272
bootstrap_type                     11
inference             CRV(country_id)
impose_null                      True
ssc                          1.198754
dtype: object
